In [1]:
!pip install pympler cloudpickle ucimlrepo

In [2]:
import os, io, json, time, warnings
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

from RuleTree.tree.TrepanClassifier import TrepanClassifier
from RuleTree.tree.TrepanClassifierOptimized import TrepanClassifierOptimized
from RuleTree.tree.RuleTreeClassifier import RuleTreeClassifier
from RuleTree.stumps.classification import  DecisionTreeStumpClassifier
from RuleTree.utils.memory_utils import deep_sklearn_sizeof, count_tree_structure

from pympler import asizeof
import cloudpickle

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
# 1. Carica Splice Junction (UCI id=69)
splice = fetch_ucirepo(id=69)
X = splice.data.features
y = splice.data.targets

# --- 2. Target: EI=0, IE=1, N=2 ---
y = y.replace({'EI': 0, 'IE': 1, 'N': 2}).astype(int).values.ravel()

# 3. One-hot encoding di tutti i 60 siti (DNA)
X = pd.get_dummies(X, dtype=int)
attributes = list(X.columns)
X = X.values.astype(np.float64)

#  4. Split stratificato 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print(f"X_train {X_train.shape} | X_test {X_test.shape}")
print(f"y_train {np.bincount(y_train)} | y_test {np.bincount(y_test)}")
print(f"Numero attributi dopo one-hot: {len(attributes)}")

X_train (2552, 287) | X_test (638, 287)
y_train [ 614  614 1324] | y_test [153 154 331]
Numero attributi dopo one-hot: 287


In [4]:
CURRENT_MAX_INTERNAL = 3
S_MIN = 1000   

MAX_LEAF =  CURRENT_MAX_INTERNAL +  1  

RESULTS_FILE = "risultati_accumulati_splice.json"

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        results = json.load(f)
    print(f"Caricati {len(results)} risultati da {RESULTS_FILE}")
else:
    results = []
    print("Nessun file precedente, parto da zero")

Caricati 24 risultati da risultati_accumulati_splice.json


In [5]:
def get_params_for(cls, oracle):
    if cls is RuleTreeClassifier:
        return dict(
            max_leaf_nodes=MAX_LEAF,
            random_state=42,
            base_stumps=DecisionTreeStumpClassifier(max_depth=1),
        )
    else:  # TrepanClassifier / TrepanClassifierOptimized
        return dict(
            estimator=oracle,
            max_internal_nodes=CURRENT_MAX_INTERNAL,
            s_min=S_MIN,
            epsilon=0.01,
            delta=0.01,
            random_state=42,
        )

In [6]:
# --- nbytes (dati puri) ---
dataset_nbytes = X_train.nbytes + y_train.nbytes

print(f"Dataset (nbytes):      {dataset_nbytes/1024/1024:.2f} MB")

Dataset (nbytes):      5.60 MB


In [7]:
# --- Oracolo: MLP come nel notebook splice-junction ---
oracle = MLPClassifier(hidden_layer_sizes=(64,), max_iter=2000, random_state=42)
oracle.fit(X_train, y_train)
print(f"Oracolo MLP accuracy test: {oracle.score(X_test, y_test):.4f}")

models = {}

for name, cls in [("base", RuleTreeClassifier),
                  ("Ottimizzato", TrepanClassifierOptimized),
                  ("Originale", TrepanClassifier)]:
    print(f"\n{'='*60}\n{name}\n{'='*60}")

    params = get_params_for(cls, oracle)
    model = cls(**params)

    t0 = time.time()
    model.fit(X_train, y_train)
    t = time.time() - t0

    asizeof_kb = asizeof.asizeof(model.root) / 1024
    deep_kb    = deep_sklearn_sizeof(model.root) / 1024

    buffer = io.BytesIO()
    cloudpickle.dump(model.root, buffer)
    cloudpickle_kb = buffer.tell() / 1024
    buffer.close()

    struct = count_tree_structure(model.root)
    acc = model.score(X_test, y_test)
    fidelity = (model.predict(X_test) == oracle.predict(X_test)).mean()

    print(f"cloudpickle:     {cloudpickle_kb:.2f} KB")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Fidelity:        {fidelity:.4f}")
    print(f"asizeof:         {asizeof_kb:.1f} KB")
    print(f"deep_sizeof:     {deep_kb:.1f} KB")
    print(f"Tempo:           {t:.2f} s")

    results.append({
        "max_internal_nodes": CURRENT_MAX_INTERNAL,
        "max_leaf_nodes":     MAX_LEAF if cls is RuleTreeClassifier else None,
        "s_min":              S_MIN,
        "name":               name,
        "accuracy":           acc,
        "fidelity":           fidelity,
        "asizeof_kb":         asizeof_kb,
        "deep_kb":            deep_kb,
        "cloudpickle_kb":     cloudpickle_kb,
        "n_nodes":            struct['n_nodes'],
        "n_stumps":           struct['n_stumps'],
        "n_leaves":           struct['n_leaves'],
        "n_conditions":       struct['n_conditions'],
        "n_rules_copies":     struct['n_rules_copies'],
        "time_s":             t,
    })

    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)

Oracolo MLP accuracy test: 0.9498

base
cloudpickle:     5.08 KB
Accuracy:        0.7194
Fidelity:        0.7132
asizeof:         25.4 KB
deep_sizeof:     23.8 KB
Tempo:           0.04 s

Ottimizzato
cloudpickle:     3.19 KB
Accuracy:        0.9185
Fidelity:        0.9075
asizeof:         13.3 KB
deep_sizeof:     12.3 KB
Tempo:           3.51 s

Originale
cloudpickle:     4.14 KB
Accuracy:        0.9185
Fidelity:        0.9075
asizeof:         15.6 KB
deep_sizeof:     14.4 KB
Tempo:           4.03 s


In [8]:
import pandas as pd, json

with open(RESULTS_FILE) as f:
    results_all = json.load(f)

df = pd.DataFrame(results_all)
df = df.drop_duplicates(subset=["max_internal_nodes", "name"], keep="last").reset_index(drop=True)

df = df[[
    "max_internal_nodes", "name",
    "accuracy", "fidelity",
    "asizeof_kb", "deep_kb", "cloudpickle_kb",
    "n_stumps", "n_leaves", "n_rules_copies", "time_s",
]]
df = df.sort_values(
    ["max_internal_nodes", "name"],
    kind="stable", na_position="last"
).reset_index(drop=True)

def fmt(x, d=3):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "--"
    return f"{x:.{d}f}"

rows = []
prev = None
for _, r in df.iterrows():
    key = r["max_internal_nodes"]
    if prev is not None and not (
        (pd.isna(prev) and pd.isna(key)) or (prev == key)
    ):
        rows.append("\\midrule")
    int_str = "--" if pd.isna(key) else str(int(key))
    rows.append(
        f"{int_str} & {r['name']:<11} & "
        f"{fmt(r['accuracy'])} & {fmt(r['fidelity'])} & "
        f"{fmt(r['asizeof_kb'])} & {fmt(r['deep_kb'])} & "
        f"{fmt(r['cloudpickle_kb'])} & "
        f"{int(r['n_stumps'])} & {int(r['n_leaves'])} & {int(r['n_rules_copies'])} & "
        f"{fmt(r['time_s'])} \\\\"
    )
    prev = key

body = "\n".join(rows)

latex = r"""\documentclass{article}
\usepackage[a4paper, left=1cm, right=2.5cm, top=2.5cm, bottom=2.5cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{makecell}

\title{Risultati memoria}
\author{Davide Catena}
\date{September 2026}

\begin{document}
\maketitle

\section{Splice-Junction Dataset}

\begin{table}[htbp]
\raggedright
\caption{Confronto TREPAN (originale e ottimizzato) e RuleTree base su Splice Junction al variare di \texttt{max\_internal\_nodes}. RuleTree usa \texttt{max\_leaf\_nodes = max\_internal\_nodes + 1}.}
\label{tab:risultati}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{c l r r r r r r r r r}
\toprule
\makecell{Int.\\nodes} & Model & Acc. & Fid. & asizeof & deep & cpickle & Stumps & Leaves & Rules & Time \\
 & & & & (KB) & (KB) & (KB) & & & & (s)  \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\end{table}

\end{document}
"""

with open("risultati_splice.tex", "w") as f:
    f.write(latex)

print(latex)

\documentclass{article}
\usepackage[a4paper, left=1cm, right=2.5cm, top=2.5cm, bottom=2.5cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{makecell}

\title{Risultati memoria}
\author{Davide Catena}
\date{September 2026}

\begin{document}
\maketitle

\section{Splice-Junction Dataset}

\begin{table}[htbp]
\raggedright
\caption{Confronto TREPAN (originale e ottimizzato) e RuleTree base su Splice Junction al variare di \texttt{max\_internal\_nodes}. RuleTree usa \texttt{max\_leaf\_nodes = max\_internal\_nodes + 1}.}
\label{tab:risultati}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{c l r r r r r r r r r}
\toprule
\makecell{Int.\\nodes} & Model & Acc. & Fid. & asizeof & deep & cpickle & Stumps & Leaves & Rules & Time \\
 & & & & (KB) & (KB) & (KB) & & & & (s)  \\
\midrule
3 & Originale   & 0.918 & 0.908 & 15.641 & 14.386 & 4.138 & 3 & 4 & 10 & 4.028 \\
3 & Ottimizzato & 0.918 & 0.908 & 13.281 & 12.276 & 3.186 & 3 & 4 & 6 & 3.505 \\
3 & base        & 0.719 &